# MLX Solar Radiation Baseline

This notebook builds a strong baseline for the solar radiation regression task using engineered time features and a `scikit-learn` pipeline.

In [60]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline


In [61]:
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train_df_1.csv'
TEST_PATH = DATA_DIR / 'test_df_1.csv'
TARGET_COL = 'Radiation'
ID_COL = 'ID'
SUBMISSION_TARGET_COL = 'TARGET'  
RANDOM_STATE = 42


In [62]:
TRAIN_PATH = 'train_df_1.csv'
TEST_PATH = 'test_df_1.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
display(train_df.head())
display(test_df.head())

train shape: (20004, 12)
test shape: (3334, 11)


,ID,UNIXTime,Data,Time,Radiation,Temperature,Pressure,Humidity,WindDirection(Degrees),Speed,TimeSunRise,TimeSunSet
0,1,1477545019,26-10-2016,19:10:19,4.294954,48.0,30.41,93.0,124.57,11.25,06:21:00,17:52:00
1,2,1477379721,24-10-2016,21:15:21,4.439169,47.0,30.47,81.0,148.63,5.62,06:21:00,17:53:00
2,3,1477689619,28-10-2016,11:20:19,483.267515,51.0,30.42,100.0,19.15,3.37,06:22:00,17:51:00
3,4,1476002422,08-10-2016,22:40:22,5.950979,50.0,30.43,90.0,189.05,5.62,06:15:00,18:05:00
4,5,1477146317,22-10-2016,04:25:17,-8.324804,49.0,NaN,102.0,117.63,4.50,06:20:00,17:54:00


,ID,UNIXTime,Data,Time,Temperature,Pressure,Humidity,WindDirection(Degrees),Speed,TimeSunRise,TimeSunSet
0,1,1479500121,18-11-2016,10:15:21,53.0,30.44,NaN,75.85,6.750000,06:33:00,17:43:00
1,2,1482676595,25-12-2016,04:36:35,46.0,NaN,31.0,173.63,NaN,06:55:00,17:51:00
2,3,1478672401,08-11-2016,20:20:01,48.0,30.46,99.0,220.98,1.120000,06:27:00,17:45:00
3,4,1483252803,31-12-2016,20:40:03,43.0,NaN,88.0,208.21,16.364613,06:57:00,17:54:00
4,5,1472903703,03-09-2016,01:55:03,52.0,30.42,96.0,163.78,4.500000,06:08:00,18:36:00


In [63]:
import numpy as np
import pandas as pd

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    # Interpolate time-series weather columns in timestamp order, then restore row order.
    sort_order = data['UNIXTime'].sort_values().index
    weather_cols = ['Pressure', 'Humidity', 'Speed']
    data.loc[sort_order, weather_cols] = (
        data.loc[sort_order, weather_cols]
        .interpolate(method='linear', limit_direction='both')
    )

    data['Data'] = pd.to_datetime(data['Data'], format='%d-%m-%Y', errors='coerce')
    data['Time'] = pd.to_datetime(data['Time'], format='%H:%M:%S', errors='coerce')
    data['TimeSunRise'] = pd.to_datetime(data['TimeSunRise'], format='%H:%M:%S', errors='coerce')
    data['TimeSunSet'] = pd.to_datetime(data['TimeSunSet'], format='%H:%M:%S', errors='coerce')

    data['month'] = data['Data'].dt.month
    data['day'] = data['Data'].dt.day
    data['dayofweek'] = data['Data'].dt.dayofweek
    data['dayofyear'] = data['Data'].dt.dayofyear
    data['weekofyear'] = data['Data'].dt.isocalendar().week.astype('float64')

    data['hour'] = data['Time'].dt.hour
    data['minute'] = data['Time'].dt.minute
    data['second'] = data['Time'].dt.second
    data['time_minutes'] = data['hour'] * 60 + data['minute'] + data['second'] / 60.0

    data['sunrise_hour'] = data['TimeSunRise'].dt.hour
    data['sunrise_minute'] = data['TimeSunRise'].dt.minute
    data['sunset_hour'] = data['TimeSunSet'].dt.hour
    data['sunset_minute'] = data['TimeSunSet'].dt.minute
    data['sunrise_minutes'] = data['sunrise_hour'] * 60 + data['sunrise_minute']
    data['sunset_minutes'] = data['sunset_hour'] * 60 + data['sunset_minute']
    data['daylight_minutes'] = data['sunset_minutes'] - data['sunrise_minutes']
    data['minutes_since_sunrise'] = data['time_minutes'] - data['sunrise_minutes']
    data['minutes_to_sunset'] = data['sunset_minutes'] - data['time_minutes']
    data['solar_noon_minutes'] = (data['sunrise_minutes'] + data['sunset_minutes']) / 2.0
    data['distance_from_noon'] = (data['time_minutes'] - data['solar_noon_minutes']).abs()

    safe_daylight = data['daylight_minutes'].replace(0, np.nan)
    data['solar_progress'] = data['minutes_since_sunrise'] / safe_daylight
    data['solar_progress_clipped'] = data['solar_progress'].clip(0, 1)
    data['midday_peak_feature'] = 1 - (data['solar_progress_clipped'] - 0.5).abs() * 2
    data['midday_peak_feature'] = data['midday_peak_feature'].clip(lower=0)

    data['is_daylight'] = (
        (data['time_minutes'] >= data['sunrise_minutes'])
        & (data['time_minutes'] <= data['sunset_minutes'])
    ).astype('float64')

    for col, period in [('hour', 24), ('month', 12), ('dayofweek', 7), ('dayofyear', 365)]:
        angle = 2 * np.pi * data[col] / period
        data[f'{col}_sin'] = np.sin(angle)
        data[f'{col}_cos'] = np.cos(angle)

    direction = np.deg2rad(data['WindDirection(Degrees)'])
    data['wind_dir_sin'] = np.sin(direction)
    data['wind_dir_cos'] = np.cos(direction)

    data['temp_humidity_interaction'] = data['Temperature'] * data['Humidity']
    data['pressure_humidity_interaction'] = data['Pressure'] * data['Humidity']
    data['wind_speed_direction_interaction'] = data['Speed'] * data['WindDirection(Degrees)']
    data['temp_solar_progress'] = data['Temperature'] * data['solar_progress_clipped']
    data['humidity_solar_progress'] = data['Humidity'] * data['solar_progress_clipped']
    data['speed_solar_progress'] = data['Speed'] * data['solar_progress_clipped']
    data['temp_midday_peak'] = data['Temperature'] * data['midday_peak_feature']
    data['humidity_midday_peak'] = data['Humidity'] * data['midday_peak_feature']

    drop_cols = ['Data', 'Time', 'TimeSunRise', 'TimeSunSet']
    return data.drop(columns=drop_cols)


In [64]:
train_features = add_time_features(train_df)
test_features = add_time_features(test_df)

X = train_features.drop(columns=[TARGET_COL])
y = train_features[TARGET_COL]
X_test = test_features.copy()

print('feature matrix:', X.shape)
print('prediction matrix:', X_test.shape)
display(X.head())

feature matrix: (20004, 49)
prediction matrix: (3334, 49)


,ID,UNIXTime,Temperature,Pressure,Humidity,WindDirection(Degrees),Speed,month,day,dayofweek,...,wind_dir_sin,wind_dir_cos,temp_humidity_interaction,pressure_humidity_interaction,wind_speed_direction_interaction,temp_solar_progress,humidity_solar_progress,speed_solar_progress,temp_midday_peak,humidity_midday_peak
0,1,1477545019,48.0,30.41,93.0,124.57,11.25,10,26,2,...,0.823434,-0.567413,4464.0,2828.13,1401.4125,48.000000,93.000000,11.250000,0.00000,0.000000
1,2,1477379721,47.0,30.47,81.0,148.63,5.62,10,24,0,...,0.520563,-0.853823,3807.0,2468.07,835.3006,47.000000,81.000000,5.620000,0.00000,0.000000
2,3,1477689619,51.0,30.42,100.0,19.15,3.37,10,28,4,...,0.328042,0.944663,5100.0,3042.00,64.5355,22.081495,43.297049,1.459111,44.16299,86.594098
3,4,1476002422,50.0,30.43,90.0,189.05,5.62,10,8,5,...,-0.157296,-0.987551,4500.0,2738.70,1062.4610,50.000000,90.000000,5.620000,0.00000,0.000000
4,5,1477146317,49.0,30.40,102.0,117.63,4.50,10,22,5,...,0.885961,-0.463760,4998.0,3100.80,529.3350,0.000000,0.000000,0.000000,0.00000,0.000000


In [66]:
numeric_features = [col for col in X.columns if col != ID_COL]

preprocessor = ColumnTransformer(
    transformers=[('num', 'passthrough', numeric_features)],
    remainder='drop',
    verbose_feature_names_out=False,
)
try:
    preprocessor.set_output(transform='pandas')
except AttributeError:
    pass

def build_base_model(random_state: int = RANDOM_STATE):
    try:
        from catboost import CatBoostRegressor

        model = CatBoostRegressor(
            loss_function='RMSE',
            eval_metric='RMSE',
            iterations=5000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=5.0,
            subsample=0.85,
            colsample_bylevel=0.8,
            min_data_in_leaf=20,
            nan_mode='Min',
            random_seed=random_state,
            verbose=False,
        )
        model_name = 'CatBoostRegressor'
    except ImportError:
        model = HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.03,
            max_iter=1200,
            max_depth=10,
            min_samples_leaf=12,
            l2_regularization=0.2,
            random_state=random_state,
        )
        model_name = 'HistGradientBoostingRegressor'

    return model, model_name

class DaylightAwareRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_pipeline, clip_negative=True, zero_out_night=True, train_daylight_only=True):
        self.base_pipeline = base_pipeline
        self.clip_negative = clip_negative
        self.zero_out_night = zero_out_night
        self.train_daylight_only = train_daylight_only

    def fit(self, X, y):
        X_fit = X
        y_fit = y

        if self.train_daylight_only and isinstance(X, pd.DataFrame) and 'is_daylight' in X.columns:
            daylight_mask = X['is_daylight'].fillna(0).to_numpy() >= 0.5
            X_fit = X.loc[daylight_mask]
            y_fit = y.loc[daylight_mask] if hasattr(y, 'loc') else y[daylight_mask]

        self.model_ = clone(self.base_pipeline)
        self.model_.fit(X_fit, y_fit)
        return self

    def predict(self, X):
        preds = self.model_.predict(X)
        preds = np.asarray(preds, dtype=float)

        if self.zero_out_night and isinstance(X, pd.DataFrame) and 'is_daylight' in X.columns:
            night_mask = X['is_daylight'].fillna(0).to_numpy() < 0.5
            preds[night_mask] = 0.0

        if self.clip_negative:
            preds = np.clip(preds, 0.0, None)

        return preds

base_model, model_name = build_base_model()

base_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', base_model),
])

pipeline = DaylightAwareRegressor(base_pipeline=base_pipeline)

print('Using model:', model_name)
pipeline

Using model: CatBoostRegressor


,base_pipeline,Pipeline(step...bose=False))])
,clip_negative,True
,zero_out_night,True
,train_daylight_only,True
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [67]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

pipeline.fit(X_train, y_train)
valid_preds = pipeline.predict(X_valid)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))

print(f'Validation RMSE: {valid_rmse:.5f}')

Validation RMSE: 80.02649


In [68]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_mse_scores = -cross_val_score(
    pipeline,
    X,
    y,
    scoring='neg_mean_squared_error',
    cv=cv,
    n_jobs=-1,
)
cv_scores = np.sqrt(cv_mse_scores)

print('Fold RMSE:', np.round(cv_scores, 5))
print(f'Mean CV RMSE: {cv_scores.mean():.5f}')
print(f'Std CV RMSE: {cv_scores.std():.5f}')

Fold RMSE: [79.83436 76.29144 73.41776 76.23017 81.65779]
Mean CV RMSE: 77.48630
Std CV RMSE: 2.91456


In [ ]:
pipeline.fit(X, y)
test_preds = pipeline.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    SUBMISSION_TARGET_COL: test_preds,
})

submission_path = DATA_DIR / 'submission_baseline.csv'
submission.to_csv(submission_path, index=False)

print(f'Saved: {submission_path.resolve()}')
display(submission.head())